# Assignment № 3. Обработка естественного языка (NLP)

In [ ]:
%pip install -q datasets "transformers<5" sentencepiece rouge_score fasttext-predict

In [ ]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay

nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("vader_lexicon")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Задание 1. Разработка и сравнительная оценка моделей анализа тональности текста

Входные данные: предложение или отзыв пользователя.
Выходные данные: negative, neutral или positive.

Три подхода: словарь (VADER), TF-IDF + Logistic Regression, Transformer-модель.

### Шаг 1. Подготовка набора данных

- Выбрать размеченный набор данных для анализа тональности, содержащий три класса.
- Использовать не менее 3 000 размеченных текстов, включая минимум 500 примеров каждого класса.
- Проанализировать распределение классов тональности.
- Удалить дублирующиеся записи, которые могут привести к утечке данных.
- Разделить набор данных на обучающую (70%), валидационную (15%) и тестовую (15%) выборки, применяя стратификацию при необходимости.

Датасет: TweetEval Sentiment (`cardiffnlp/tweet_eval`), часть test. Метки: 0 = negative, 1 = neutral, 2 = positive.

In [ ]:
from datasets import load_dataset

# загружаем датасет
data = load_dataset("cardiffnlp/tweet_eval", "sentiment")
df = data["test"].to_pandas()
print("Всего твитов:", len(df))
print(df["label"].value_counts().sort_index())

# удаляем дубликаты
df = df.drop_duplicates(subset="text")

# берём по 1000 твитов каждого класса
df = df.groupby("label").sample(n=1000, random_state=SEED)
print("\nПосле выборки:", len(df))
print(df["label"].value_counts().sort_index())

# делим 70 / 15 / 15 со стратификацией
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
print("\ntrain:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

**Распределение классов.** В исходных данных классы несбалансированы: neutral 5937, negative 3972, positive 2375. После выборки по 1000 твитов каждого класса — 3000 текстов, классы равны. Дубликатов нет. Деление: train 2100, validation 450, test 450.

### Шаг 2. Предварительная обработка текста

Реализовать и сравнить подходящие методы предварительной обработки:

- Токенизация.
- Приведение текста к нижнему регистру.
- Удаление несущественных символов.
- Удаление стоп-слов, если это целесообразно.
- Лемматизация.

Объяснить, почему чрезмерная предварительная обработка может отрицательно влиять на словарные методы и Transformer-модели, особенно при удалении отрицаний или других признаков эмоциональной окраски.

In [ ]:
lemmatizer = WordNetLemmatizer()

# стоп-слова, но без отрицаний — они меняют тональность
stop_words = set(stopwords.words("english")) - {"not", "no", "nor", "never"}

# простая обработка: нижний регистр + токенизация
def simple_preprocess(text):
    tokens = word_tokenize(text.lower())
    return " ".join(tokens)

# полная обработка: все пункты шага
def full_preprocess(text):
    # нижний регистр
    text = text.lower()
    # удаляем ссылки, @user и лишние символы
    text = re.sub(r"http\S+|@\w+|[^a-z\s']", " ", text)
    # токенизация
    tokens = word_tokenize(text)
    # удаляем стоп-слова
    tokens = [t for t in tokens if t not in stop_words]
    # лемматизация
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

for part in [train_df, val_df, test_df]:
    part["simple"] = part["text"].apply(simple_preprocess)
    part["full"] = part["text"].apply(full_preprocess)

print("Исходный:", train_df["text"].iloc[0])
print("Простая:", train_df["simple"].iloc[0])
print("Полная:", train_df["full"].iloc[0])

In [ ]:
# сравниваем: одна и та же модель на двух вариантах обработки, оценка на валидации
scores = {}
for column in ["simple", "full"]:
    vectorizer = TfidfVectorizer()
    X_tr = vectorizer.fit_transform(train_df[column])
    X_v = vectorizer.transform(val_df[column])
    model = LogisticRegression(max_iter=1000).fit(X_tr, train_df["label"])
    scores[column] = f1_score(val_df["label"], model.predict(X_v), average="macro")
    print(column, "-> macro-F1 на валидации:", round(scores[column], 4))

# в шаге 4 используем лучший вариант
best_column = max(scores, key=scores.get)
print("Лучший вариант:", best_column)

**Сравнение.** Простая обработка дала macro-F1 0,609 на валидации, полная — 0,597. В коротких твитах каждое слово важно, и удаление слов немного ухудшает результат. Поэтому в шаге 4 используется простая обработка.

**Почему чрезмерная обработка вредит словарным методам и Transformer.**
- Отрицания (`not`, `never`, `n't`) переворачивают смысл: «not good» без стоп-слов превращается в «good».
- VADER учитывает `!!!`, КАПС и смайлики — после нижнего регистра и удаления символов этих сигналов нет.
- Transformer обучен на обычном тексте; после удаления слов и лемматизации текст становится непохожим на то, что модель видела.

Поэтому VADER и Transformer получают исходный текст.

### Шаг 3. Анализ тональности на основе словаря

- Реализовать классификатор тональности с использованием VADER или подходящего словаря эмоционально окрашенных слов.
- Рассчитать оценки тональности (Sentiment Scores).
- Классифицировать тексты по трём категориям тональности.
- При необходимости настройки пороговых значений использовать только валидационную выборку.
- Оценить качество полученных предсказаний.

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# оценка тональности: compound от -1 до 1, на исходном тексте
val_df["vader_score"] = val_df["text"].apply(lambda t: sia.polarity_scores(t)["compound"])
test_df["vader_score"] = test_df["text"].apply(lambda t: sia.polarity_scores(t)["compound"])

# оценка -> класс
def vader_label(score, threshold):
    if score >= threshold:
        return 2
    if score <= -threshold:
        return 0
    return 1

# подбираем порог только на валидации
best_threshold, best_f1 = 0, 0
for threshold in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]:
    pred = val_df["vader_score"].apply(lambda s: vader_label(s, threshold))
    f1 = f1_score(val_df["label"], pred, average="macro")
    print("порог", threshold, "-> macro-F1 на валидации:", round(f1, 4))
    if f1 > best_f1:
        best_threshold, best_f1 = threshold, f1

print("\nЛучший порог:", best_threshold)

# классифицируем тест и оцениваем
test_df["vader_pred"] = test_df["vader_score"].apply(lambda s: vader_label(s, best_threshold))
print("VADER на тесте: accuracy =", round(accuracy_score(test_df["label"], test_df["vader_pred"]), 4),
      "| macro-F1 =", round(f1_score(test_df["label"], test_df["vader_pred"], average="macro"), 4))

### Шаг 4. Модель машинного обучения

- Преобразовать тексты в числовые векторы с помощью TF-IDF.
- Обучить классификатор Logistic Regression или Multinomial Naive Bayes.
- Проверить не менее двух конфигураций гиперпараметров.
- Оценить модель на валидационной выборке и выбрать лучшую конфигурацию.

In [ ]:
# TF-IDF: обучаем только на train
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_df[best_column])
X_val = vectorizer.transform(val_df[best_column])
X_test = vectorizer.transform(test_df[best_column])

# Logistic Regression с разными C
best_model, best_f1 = None, 0
for C in [0.1, 1, 10]:
    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_train, train_df["label"])
    f1 = f1_score(val_df["label"], model.predict(X_val), average="macro")
    print("C =", C, "-> macro-F1 на валидации:", round(f1, 4))
    if f1 > best_f1:
        best_model, best_f1, best_C = model, f1, C

print("\nЛучшая конфигурация: C =", best_C)
test_df["lr_pred"] = best_model.predict(X_test)

### Шаг 5. Модель на основе Transformer

- Выбрать предварительно обученную Transformer-модель для анализа тональности из Hugging Face, поддерживающую язык выбранного датасета и три целевых класса.
- Применить модель к тестовой выборке без дополнительного дообучения.
- При необходимости привести выходные метки модели к меткам используемого набора данных.
- Сравнить результаты с традиционными подходами.

Модель: `cardiffnlp/twitter-roberta-base-sentiment-latest` (английский, классы negative / neutral / positive).

In [ ]:
import torch
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0 if torch.cuda.is_available() else -1,
)

# предсказания на тесте
results = classifier(test_df["text"].tolist(), truncation=True)

# метки модели -> метки датасета
label_map = {"negative": 0, "neutral": 1, "positive": 2}
test_df["tr_pred"] = [label_map[r["label"]] for r in results]
print(results[:3])

del classifier

### Шаг 6. Оценка качества моделей

Рассчитать следующие метрики:

- Accuracy — точность классификации.
- Precision (macro) — макроусреднённая точность положительных предсказаний.
- Recall (macro) — макроусреднённая полнота.
- F1-score (macro) — макроусреднённая F1-мера.
- Confusion Matrix — матрица ошибок классификации.

Построить сравнительную таблицу с результатами оценки всех трёх подходов.

In [ ]:
y_test = test_df["label"]
models = {"VADER": "vader_pred", "TF-IDF + LR": "lr_pred", "Transformer": "tr_pred"}

# сравнительная таблица
rows = []
for name, column in models.items():
    pred = test_df[column]
    rows.append({
        "модель": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision (macro)": precision_score(y_test, pred, average="macro"),
        "recall (macro)": recall_score(y_test, pred, average="macro"),
        "F1 (macro)": f1_score(y_test, pred, average="macro"),
    })
pd.DataFrame(rows).round(4)

In [ ]:
# матрицы ошибок
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, column) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, test_df[column], display_labels=["neg", "neu", "pos"], ax=ax, colorbar=False)
    ax.set_title(name)
plt.show()

**Результаты на тесте:**

| модель | accuracy | macro-F1 |
|---|---|---|
| VADER | 0,531 | 0,528 |
| TF-IDF + LR | 0,611 | 0,605 |
| Transformer | 0,718 | 0,718 |

- Лучший порог VADER на валидации — 0,05.
- Лучшая конфигурация Logistic Regression — C = 1.
- Transformer лучше остальных подходов по всем метрикам.

### Шаг 7. Анализ ошибок

- Найти не менее пяти неправильно классифицированных отзывов.
- Сравнить предсказания различных моделей.
- Объяснить возможные причины ошибок: отрицания, сарказм, многозначность слов и предметно-специфические выражения.
- Определить наиболее эффективный подход и обосновать вывод экспериментальными результатами.

Исследовательский вопрос: обеспечивают ли предварительно обученные Transformer-модели существенное улучшение качества анализа тональности по сравнению со словарными методами и традиционным машинным обучением на выбранном наборе данных?

In [ ]:
# твиты, где ошиблась хотя бы одна модель
wrong = test_df[
    (test_df["vader_pred"] != test_df["label"])
    | (test_df["lr_pred"] != test_df["label"])
    | (test_df["tr_pred"] != test_df["label"])
]
print("Твитов с ошибкой хотя бы одной модели:", len(wrong))

pd.set_option("display.max_colwidth", 150)
wrong[["text", "label", "vader_pred", "lr_pred", "tr_pred"]].sample(n=10, random_state=SEED)

**Разбор ошибок** (примеры из таблицы; 0 = negative, 1 = neutral, 2 = positive):

1. «this picture drawn by muhammad ali is KILLING ME» — на самом деле positive (восторг), VADER и Transformer ставят negative. **Многозначность:** «killing me» в сленге означает «очень нравится».
2. «When Trump jeopardizes their social security ... imma laugh.» — negative, VADER ставит positive из-за слова «laugh». **Сарказм / злорадство:** словарь видит только отдельное слово.
3. «For no reason because animal testing doesn't translate to humans anyway» — neutral, все три модели ставят negative. **Отрицания** (`no`, `doesn't`) модели принимают за негатив, хотя это просто утверждение.
4. «Review of @user – Ashley Graham POV Boxing Defeat» — neutral, VADER ставит negative из-за слова «Defeat». **Предметное выражение:** название видео, а не эмоция.
5. «@user #Assad regime, #Hezbollah and #YPG will capture #Al_Bab...» — negative, VADER ставит neutral. **Предметная тема** (война) без эмоциональных слов: словарь их не находит, а TF-IDF и Transformer справились.

**Сравнение моделей.** VADER ошибается, когда смысл не в отдельных словах (сарказм, сленг, тема). TF-IDF + LR лучше, но не понимает порядок и контекст. Transformer учитывает контекст и ошибается реже, но тоже путается на сарказме и спорных твитах.

**Наиболее эффективный подход** — Transformer: macro-F1 0,718 против 0,605 у TF-IDF + LR и 0,528 у VADER.

**Ответ на исследовательский вопрос:** да, предобученная Transformer-модель даёт существенное улучшение — на 11 п.п. выше TF-IDF + LR и на 19 п.п. выше VADER.

# Задание 2. Разработка и оценка методов автоматического суммирования текста

Входные данные: пять англоязычных текстовых документов по взаимосвязанной тематике.

Выходные данные:
1. Экстрактивное резюме, содержащее по три наиболее важных предложения из каждого документа.
2. Объединённое экстрактивное резюме из шести предложений, выбранных из пятнадцати ранее извлечённых.
3. Абстрактивное резюме объёмом примерно 100–150 слов, охватывающее содержание всех пяти документов.

### Шаг 1. Сбор данных

- Выбрать пять тематически связанных документов, предпочтительно новостных статей, научных аннотаций или технических отчётов.
- Каждый документ должен содержать не менее 250 слов.
- Указать источник и количество слов для каждого документа.
- Подготовить краткое эталонное резюме объёмом 100–150 слов, написанное человеком, либо использовать датасет с подходящими эталонными резюме.

Документы: пять статей об автоматическом обнаружении музыкального плагиата (`article1.txt` … `article5.txt`). Эталонное резюме: `reference.txt` (черновик подготовлен с помощью ИИ-ассистента до запуска моделей и сверен со статьями).

In [ ]:
import os
import urllib.request

# если файлов нет рядом (например, в Colab) — скачиваем с GitHub
url = "https://raw.githubusercontent.com/DIas-code/AITU/main/Programing%20for%20data%20analysis/assignment3/"
for name in ["article1.txt", "article2.txt", "article3.txt", "article4.txt", "article5.txt", "reference.txt"]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(url + name, name)

documents = []
for i in range(1, 6):
    with open(f"article{i}.txt", encoding="utf-8") as f:
        documents.append(f.read())

with open("reference.txt", encoding="utf-8") as f:
    reference = f.read()

sources = [
    "Music plagiarism detection system (melody extraction, PCM -> MIDI)",
    "Music Plagiarism Detection Using Melody Databases",
    "Music Plagiarism Detector (MFCC, entropy, energy)",
    "Music Plagiarism Detector, NCDMC-2017 (введение, sampling)",
    "Music Plagiarism Detector, NCDMC-2017 (rhythm, melody)",
]
for i in range(5):
    print(f"article{i + 1}: {len(documents[i].split())} слов | {sources[i]}")
print("\nЭталонное резюме:", len(reference.split()), "слов")

### Шаг 2. Предварительная обработка текста

- Выполнить токенизацию на уровне предложений.
- Удалить лишние пробелы и символы форматирования.
- Применить подходящую предварительную обработку для расчёта значимости предложений.
- Сохранить исходные предложения для формирования итоговых резюме.

In [ ]:
# удаляем служебные символы из PDF, колонтитулы, ссылки [12], маркеры списка и лишние пробелы
def clean(text):
    text = text.replace("\x02", "")
    text = text.replace("Music Plagiarism Detection Using Melody Databases 685", " ")
    text = text.replace("National Conference On Discrete Mathematics & Computing (NCDMC-2017) 9 | Page", " ")
    text = re.sub(r"\[\d+\]", " ", text)
    text = text.replace("•", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# токенизация на предложения; исходные предложения сохраняем
doc_sentences = []
for doc in documents:
    sentences = sent_tokenize(clean(doc))
    # убираем заголовки и обрывки (меньше 5 слов)
    sentences = [s for s in sentences if len(s.split()) >= 5]
    # убираем повторы (в article3 абзац Abstract записан дважды)
    sentences = list(dict.fromkeys(sentences))
    doc_sentences.append(sentences)
    print(len(sentences), "предложений")

# для расчёта значимости: нижний регистр и без стоп-слов (делает TfidfVectorizer в шаге 3)

### Шаг 3. Экстрактивное суммирование

Реализовать экстрактивный подход с использованием оценки значимости предложений на основе TF-IDF или TextRank.

- Рассчитать показатели значимости предложений.
- Выбрать три наиболее значимых предложения из каждого документа.
- Объединить пятнадцать выбранных предложений в один набор кандидатов.
- Выбрать шесть предложений для итогового объединённого резюме.
- Сохранить исходные формулировки и расположить предложения в логичном, удобном для чтения порядке.

In [ ]:
# значимость предложения = сумма TF-IDF весов его слов
def sentence_scores(sentences):
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(sentences)
    return np.asarray(X.sum(axis=1)).ravel()

# по три лучших предложения из каждого документа
candidates = []
for i, sentences in enumerate(doc_sentences):
    scores = sentence_scores(sentences)
    top3 = sorted(scores.argsort()[::-1][:3])  # sorted — порядок как в документе
    print(f"--- article{i + 1} ---")
    for j in top3:
        print("•", sentences[j])
        candidates.append(sentences[j])

print("\nКандидатов:", len(candidates))

In [ ]:
# шесть лучших из пятнадцати, в исходном порядке
scores = sentence_scores(candidates)
top6 = sorted(scores.argsort()[::-1][:6])
extractive_summary = " ".join(candidates[j] for j in top6)

print(extractive_summary)
print("\nСлов:", len(extractive_summary.split()))

### Шаг 4. Абстрактивное суммирование

Использовать предварительно обученную Transformer-модель, например BART или T5.

- Сгенерировать резюме объёмом примерно 100–150 слов.
- Учесть ограничения модели по длине входного текста, применив подходящую стратегию, например разбиение текста на фрагменты (Chunking) или иерархическое суммирование.
- Зафиксировать параметры генерации.
- Проверить, насколько корректно сформированное резюме отражает информацию исходных документов.

Модель: `facebook/bart-large-cnn`. Стратегия: иерархическое суммирование — сначала резюме каждого документа, затем резюме этих резюме.

In [ ]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0 if torch.cuda.is_available() else -1,
)

# 1) резюме каждого документа (длинный текст обрезается до 1024 токенов)
doc_summaries = []
for doc in documents:
    result = summarizer(clean(doc), max_length=100, min_length=40, num_beams=4, do_sample=False, truncation=True)
    doc_summaries.append(result[0]["summary_text"])

# 2) резюме пяти резюме — итог
abstractive_summary = summarizer(
    " ".join(doc_summaries), max_length=200, min_length=130, num_beams=4, do_sample=False, truncation=True
)[0]["summary_text"]

print(abstractive_summary)
print("\nСлов:", len(abstractive_summary.split()))

del summarizer

**Проверка абстрактивного резюме по статьям:**
- Верно перенесены факты: потери от плагиата в Корее больше 2 млрд долларов (2009); 1,4 млрд проданных треков в 2008 году и больше 1,8 млрд позже; модель сходства с выравниванием и сдвигом; индексирование признаков мелодии.
- Выдуманных фактов нет — все предложения взяты из статей почти дословно.
- **Пропущено:** извлечение мелодии из PCM в MIDI (статья 1), признаки MFCC, энтропия и энергия (статья 3), виды плагиата — сэмплинг, ритм, мелодия (статьи 4–5).
- **Смешаны источники:** «We propose a novel similarity model...» — это статья 2, а «Our project set out to be a beacon of truth...» — статья 3. В общем резюме непонятно, кто такие «we».
- **Грамматическая ошибка:** «...more than 2 billion dollar (2009) We propose...» — нет точки, два предложения склеены.

### Шаг 5. Количественная оценка качества

Оценить итоговые экстрактивное и абстрактивное резюме относительно одного и того же эталонного резюме с использованием:

- ROUGE-1.
- ROUGE-2.
- ROUGE-L.

Представить результаты в сравнительной таблице.

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rows = []
for name, summary in [("Экстрактивное", extractive_summary), ("Абстрактивное", abstractive_summary)]:
    scores = scorer.score(reference, summary)
    rows.append({
        "резюме": name,
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure,
    })
pd.DataFrame(rows).round(4)

**Результат:**
- По ROUGE-1 выше экстрактивное резюме: 0,392 против 0,343. Но оно длиннее (224 слова против 125), а ROUGE-1 растёт, когда больше слов совпадает с эталоном.
- По ROUGE-2 (0,068 против 0,060) и ROUGE-L (0,179 против 0,147) выше абстрактивное.
- У обоих ROUGE-2 низкий: эталон написан своими словами, и общих пар слов подряд мало.

### Шаг 6. Качественный анализ

Оценить оба резюме по следующим критериям:

- Полнота охвата информации.
- Связность и читаемость.
- Избыточность информации.
- Фактическая согласованность с исходными документами.
- Грамматическая корректность.

Определить не менее трёх различий между экстрактивным и абстрактивным суммированием. Рассмотреть как минимум один пример, в котором абстрактивная модель изменяет, пропускает или добавляет информацию.

Исследовательский вопрос: какой подход к суммированию обеспечивает лучший баланс между полнотой информации, читаемостью и фактической достоверностью?

| критерий | экстрактивное | абстрактивное |
|---|---|---|
| Полнота | статьи 2, 3, 4, 5; статьи 1 нет | статьи 1–4 частично; нет видов плагиата и MFCC |
| Связность и читаемость | отдельные предложения из разных статей, без связок | читается как один текст |
| Избыточность | общие фразы («Introduction Recently, the use of multimedia data...») | повторов нет |
| Фактическая точность | всё дословно из статей | факты верные, но смешаны «we» разных статей |
| Грамматика | ошибки самих статей: «In2004», «environmentscar», «litErature», пустые места после удалённых ссылок «presented in ,» | склейка «...dollar (2009) We propose» |

**Три различия:**
1. Экстрактивное берёт предложения дословно, абстрактивное генерирует текст само.
2. Экстрактивное не может сократить длинное предложение, поэтому оно длиннее (224 слова против 125).
3. Экстрактивное выбирает самые «весомые» по TF-IDF предложения, а абстрактивное может объединять мысли, но и пропускать их.

**Пример изменения информации.** BART пишет «We propose a novel similarity model...» и сразу «Our project set out to be a beacon of truth...». В статьях это два разных автора (статьи 2 и 3), а в резюме выглядит как одна работа. Кроме того, BART полностью пропустил виды плагиата (сэмплинг, ритм, мелодия), которые есть в эталоне.

**Ответ на исследовательский вопрос.**
- Экстрактивный подход лучше по фактической достоверности — ничего не меняет — и по ROUGE-1.
- Абстрактивный лучше по читаемости и по ROUGE-2 и ROUGE-L, но может смешивать источники и пропускать темы.

Лучший баланс для научных статей — экстрактивный подход: точность важнее гладкости текста.

# Задание 3. Лексический анализ и автоматическая классификация музыкальных жанров на основе текстов песен

Входные данные: тексты песен четырёх жанров.

Выходные данные: сравнительный лексический анализ жанров; частотные, редкие и тематические слова; классификация жанров; оценка качества моделей.

### Шаг 1. Сбор данных

- Выбрать датасет, содержащий тексты песен и метки музыкальных жанров.
- Выбрать четыре жанра, например Pop, Rock, Hip-Hop и Country.
- Использовать не менее 100 песен каждого жанра (минимум 400 песен).
- Использовать один язык для всего корпуса, предпочтительно английский.
- Удалить дублирующиеся тексты песен.
- Проанализировать распределение длины текстов и количество песен по жанрам.

Датасет: `Yegor25/lyrics_genre_dataset_medium` (Hugging Face). Жанр `rap` считаем Hip-Hop.

In [ ]:
lyrics = load_dataset("Yegor25/lyrics_genre_dataset_medium", split="train")

# только английские песни с одним жанром
lyrics = lyrics.filter(lambda x: x["language"] == "english" and len(x["genre"]) == 1)

# по 400 случайных песен каждого жанра
genres = {"pop": "Pop", "rock": "Rock", "rap": "Hip-Hop", "country": "Country"}
parts = []
for raw_name, name in genres.items():
    part = lyrics.filter(lambda x: x["genre"][0].lower() == raw_name)
    part = part.shuffle(seed=SEED).select(range(400)).to_pandas()
    part["genre"] = name
    parts.append(part[["lyrics", "genre"]])
songs = pd.concat(parts, ignore_index=True)

# проверка языка: метка "english" бывает ошибочной (есть песни на португальском);
# в английском тексте много английских стоп-слов (the, and, you), в других языках — мало
stop_words_all = set(stopwords.words("english"))

def is_english(text):
    words = text.lower().split()
    if len(words) == 0:
        return False
    return sum(w in stop_words_all for w in words) / len(words) > 0.25

print("Не на английском:", (~songs["lyrics"].apply(is_english)).sum())
songs = songs[songs["lyrics"].apply(is_english)]

# удаляем дубликаты и оставляем по 300 песен на жанр
songs = songs.drop_duplicates(subset="lyrics")
songs = songs.groupby("genre").sample(n=300, random_state=SEED).reset_index(drop=True)
print(songs["genre"].value_counts())

In [ ]:
# длина текстов по жанрам
songs["n_words"] = songs["lyrics"].str.split().str.len()
songs.groupby("genre")["n_words"].describe().round(0)

**Данные.** 73 песни с меткой «english» оказались не на английском (португальский и др.) и удалены. Дубликатов нет. В каждом жанре по 300 песен — классы равны.

**Длина текстов:** Hip-Hop почти вдвое длиннее остальных — медиана 422 слова против 188–226 у Country, Pop и Rock. Поэтому жанры сравниваем по относительным частотам.

### Шаг 2. Предварительная обработка текста

Реализовать:

- Приведение текста к нижнему регистру.
- Токенизацию.
- Удаление знаков препинания и ненужных символов.
- Удаление стоп-слов.
- Лемматизацию.

Объяснить влияние предварительной обработки на лексические характеристики текстов песен.

In [ ]:
def preprocess_lyrics(text):
    # нижний регистр
    text = text.lower()
    # убираем пометки [Chorus], [Verse 1]
    text = re.sub(r"\[.*?\]", " ", text)
    # токенизация
    tokens = word_tokenize(text)
    # убираем знаки препинания, цифры и обрывки короче 3 букв (na от gonna, im от I'm)
    tokens = [t for t in tokens if t.isalpha() and len(t) > 2]
    # убираем стоп-слова
    tokens = [t for t in tokens if t not in stop_words_all]
    # лемматизация
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

songs["tokens"] = songs["lyrics"].apply(preprocess_lyrics)
songs["clean"] = songs["tokens"].apply(" ".join)

print("Слов в песне до обработки:", round(songs["n_words"].mean()))
print("Слов в песне после обработки:", round(songs["tokens"].str.len().mean()))

**Влияние обработки.** В средней песне было 277 слов, осталось 130 (−53%).
- Ушли стоп-слова (I, you, the, and) — они есть во всех жанрах и жанр не различают.
- Ушли обрывки (`na` от gonna, `im` от I'm) и пометки [Chorus], [Verse].
- Формы слов объединены лемматизацией (`eyes` → `eye`).
- Но вместе с этим теряются особенности стиля: повторы, междометия (`oh`), разговорные формы.

### Шаг 3. Частотный анализ

Для каждого жанра:

- Определить 10 наиболее часто встречающихся слов.
- Рассчитать их абсолютные и относительные частоты.
- Визуализировать результаты с помощью столбчатых диаграмм.
- Сравнить особенности словарного состава различных жанров.

In [ ]:
from collections import Counter

GENRES = ["Country", "Hip-Hop", "Pop", "Rock"]

# считаем слова каждого жанра
counts = {}
for g in GENRES:
    counts[g] = Counter()
    for tokens in songs[songs["genre"] == g]["tokens"]:
        counts[g].update(tokens)

# топ-10: абсолютная и относительная частота
top10 = {}
for g in GENRES:
    total = sum(counts[g].values())
    top10[g] = pd.DataFrame(counts[g].most_common(10), columns=["слово", "абс."])
    top10[g]["отн., %"] = (top10[g]["абс."] / total * 100).round(2)
    print(g)
    print(top10[g].to_string(index=False), "\n")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, g in zip(axes, GENRES):
    ax.bar(top10[g]["слово"], top10[g]["абс."])
    ax.set_title(g)
    ax.tick_params(axis="x", rotation=60)
plt.show()

**Сравнение жанров.**
- Общие слова у всех жанров: `like`, `know`, `got`, `love`, `time`.
- **Hip-Hop:** сленг и ненормативная лексика (`nigga`, `bitch`, `shit`, `fuck`), `yeah`. Слова `love` нет в его топ-10.
- **Country:** `love` на первом месте.
- **Rock:** `never`.
- **Pop:** `take`, `feel`.

Country, Pop и Rock по топ-10 почти одинаковы.

### Шаг 4. Анализ лексических категорий

Создать словари минимум для трёх лексических категорий, например:

- Временные выражения: morning, evening, night.
- Эмоции: love, sadness, happiness, anger.
- Части тела: heart, eyes, hands.

Рассчитать частоты слов каждой категории для каждого жанра.

Нормализовать частоты категорий по общему количеству слов соответствующего жанра, чтобы обеспечить корректность сравнения.

In [ ]:
categories = {
    "время": ["morning", "evening", "night", "day", "today", "tonight", "tomorrow", "yesterday"],
    "эмоции": ["love", "sadness", "happiness", "anger", "hate", "happy", "sad", "cry", "fear", "pain"],
    "тело": ["heart", "eye", "hand", "head", "face", "lip", "body", "arm"],
}

rows = []
for g in GENRES:
    total = sum(counts[g].values())
    row = {"жанр": g}
    for name, words in categories.items():
        count = sum(counts[g][w] for w in words)
        row[name] = count
        row[name + " на 1000 слов"] = round(count / total * 1000, 2)
    rows.append(row)
pd.DataFrame(rows)

**Частоты на 1000 слов:**
- **Время** чаще всего в Pop (15,6) и Country (13,0).
- **Эмоции** — в Country (21,2) и Rock (19,8).
- **Части тела** — в Rock (18,6).
- **Hip-Hop** — меньше всех по всем трём категориям (5,1 / 9,7 / 7,9): словарь рэпа разнообразнее, такие слова встречаются реже.

Без нормализации сравнение было бы неверным: Hip-Hop вдвое длиннее, и его абсолютные числа завышены.

### Шаг 5. Редкие и характерные слова

- Найти слова, встречающиеся не более трёх раз в пределах каждого жанра.
- Рассчитать показатели TF-IDF для определения характерных слов различных жанров.
- Сравнить редкие слова со словами, имеющими высокие значения TF-IDF.
- Объяснить, почему редкие слова не всегда являются наиболее информативными признаками для определения музыкального жанра.

In [ ]:
# редкие слова: не более 3 раз в жанре
rare = {}
for g in GENRES:
    rare[g] = [w for w, c in counts[g].items() if c <= 3]
    print(g, "— редких слов:", len(rare[g]), "| примеры:", rare[g][:8])

In [ ]:
# TF-IDF по песням; характерные слова жанра — у которых средний TF-IDF в этом жанре
# больше всего превышает средний TF-IDF в остальных жанрах
vectorizer = TfidfVectorizer(min_df=5)
X = vectorizer.fit_transform(songs["clean"])
words = vectorizer.get_feature_names_out()

for g in GENRES:
    in_genre = (songs["genre"] == g).values
    difference = np.asarray(X[in_genre].mean(axis=0) - X[~in_genre].mean(axis=0)).ravel()
    top = [words[j] for j in difference.argsort()[::-1][:10]]
    print(g, "— характерные слова:", top)
    print("   из них редких:", [w for w in top if w in rare[g]])

**Редкие и характерные слова.**
- Редких слов (не больше 3 раз в жанре) тысячи: от 2570 в Country до 6241 в Hip-Hop. Это случайные слова и имена: `joshua`, `seaweed`, `whale`, `timepiece`.
- Ни одно из 10 характерных слов по TF-IDF не редкое. Характерные слова:
  - **Country:** `old`, `blue`, `town`, `country`, `sweet`;
  - **Hip-Hop:** сленг, `tryna`;
  - **Pop:** `night`, `girl`, `party`, `mirror`, `pretty`;
  - **Rock:** `end`, `lie`, `fear`, `death`, `lonely`.

**Почему редкие слова не информативны.** Слово, встреченное 1–3 раза, почти не встретится в новых песнях жанра — модель не может на него опереться. Часто это имя или случайное слово. Информативны слова, которые часто встречаются в одном жанре и редко в других — это и показывает TF-IDF.

### Шаг 6. Автоматическая классификация жанров

Разработать два подхода:

Модель 1: TF-IDF + Multinomial Naive Bayes.

Модель 2: TF-IDF + Logistic Regression.

- Разделить данные на обучающую (80%) и тестовую (20%) выборки с использованием стратификации.
- Обучать TF-IDF-преобразователь только на обучающей выборке.
- При наличии информации об исполнителях рассмотреть разделение данных по исполнителям для исследования потенциальной утечки данных.
- Обучить обе классификационные модели.

Информации об исполнителях в датасете нет, поэтому разделение по исполнителям не выполняется.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    songs["clean"], songs["genre"], test_size=0.20, stratify=songs["genre"], random_state=SEED
)

# TF-IDF обучаем только на train
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

nb = MultinomialNB().fit(X_train, y_train)
lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)

### Шаг 7. Оценка и сравнение результатов

Рассчитать:

- Accuracy.
- Macro F1-score.
- Confusion Matrix.

Сравнить эффективность двух моделей.

Проанализировать не менее пяти ошибочных предсказаний и объяснить, какие музыкальные жанры чаще всего путаются.

Обсудить, достаточно ли только текстов песен для определения музыкального жанра, учитывая такие музыкальные характеристики, как ритм, инструменты и особенности вокала.

Исследовательский вопрос: позволяют ли лексические признаки и TF-IDF-представления различать музыкальные жанры и какие лингвистические характеристики оказывают наибольшее влияние на качество классификации?

In [ ]:
predictions = {"Naive Bayes": nb.predict(X_test), "Logistic Regression": lr.predict(X_test)}

for name, pred in predictions.items():
    print(name, "| accuracy =", round(accuracy_score(y_test, pred), 4), "| macro-F1 =", round(f1_score(y_test, pred, average="macro"), 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=GENRES, ax=ax, colorbar=False)
    ax.set_title(name)
plt.show()

In [ ]:
# пять ошибок Logistic Regression
shown = 0
for i, true, pred in zip(X_test_text.index, y_test, predictions["Logistic Regression"]):
    if true != pred and shown < 5:
        print("верно:", true, "| предсказано:", pred)
        print(songs.loc[i, "lyrics"][:200].replace("\n", " / "), "...\n")
        shown += 1

**Сравнение моделей:**
- Naive Bayes: accuracy 0,575, macro-F1 0,542.
- Logistic Regression: accuracy 0,550, macro-F1 0,552.

Модели почти равны. NB чаще угадывает Hip-Hop и Country, но плохо распознаёт Pop. LR ровнее по всем жанрам, поэтому у неё выше macro-F1.

**Что путается чаще всего** (по матрицам ошибок):
- **Pop** с Rock и Country: у LR 22 песни Pop названы Rock и 17 — Country; у NB 26 песен Pop названы Country.
- **Rock** с Country (15–17 песен).
- **Hip-Hop** распознаётся лучше всех: NB — 57 из 60, LR — 48 из 60. У него свой словарь.

**Пять ошибок выше:**
- кантри-песня про охоту и «truck» → Pop;
- поп-песня со сленгом и матом («feds», «bricks») → Hip-Hop — по тексту это рэп;
- рок-песня «Another day, another dawn... rearview» → Country;
- рок-песня про «love», «pop stars» → Pop;
- диско-поп «boogie... we've got the beat» → Hip-Hop.

Слова о любви, дороге, ночи одинаковы в Pop, Rock и Country.

**Достаточно ли текста?** Нет. Жанр определяют ритм, темп, инструменты (гитары, банджо, биты) и манера вокала (рэп-читка, пение). Одна и та же песня о любви может быть поп, рок или кантри. По тексту хорошо отделяется только Hip-Hop.

**Ответ на исследовательский вопрос.** Лексические признаки и TF-IDF различают жанры частично: macro-F1 около 0,55 при 4 классах (случайное угадывание — 0,25). Сильнее всего влияют:
1. сленг и ненормативная лексика — выделяют Hip-Hop;
2. тематические слова (`old`, `town` в Country, `fear`, `death` в Rock, `party`, `girl` в Pop);
3. длина текста.

# Задание 4. Многоязычная идентификация языка и анализ устойчивости моделей

Входные данные: короткий текст (одно слово, одно предложение или несколько предложений).

Выходные данные: предсказанная метка языка и оценка уверенности модели.

Система распознаёт 6 языков: казахский, русский, английский, испанский, португальский, немецкий.

### Шаг 1. Подготовка набора данных

- Использовать готовый многоязычный датасет, например Tatoeba или WiLI-2018.
- Выбрать от шести до восьми языков, включая казахский, русский и английский.
- Добавить как минимум одну пару близкородственных языков, например испанский и португальский.
- Собрать не менее 500 текстовых примеров для каждого языка.
- Удалить полные дубликаты.
- Разделить датасет на обучающую (80%) и тестовую (20%) выборки.

Датасет: WiLI-2018 (zenodo.org/record/841984), абзацы из Википедии. Близкородственная пара: испанский и португальский.

In [ ]:
import io
import zipfile

# скачиваем WiLI-2018
if not os.path.exists("wili-2018"):
    response = urllib.request.urlopen("https://zenodo.org/record/841984/files/wili-2018.zip")
    zipfile.ZipFile(io.BytesIO(response.read())).extractall("wili-2018")

# читаем тексты и метки (обе части WiLI)
texts, labels = [], []
for split in ["train", "test"]:
    texts += open(f"wili-2018/x_{split}.txt", encoding="utf-8").read().splitlines()
    labels += open(f"wili-2018/y_{split}.txt", encoding="utf-8").read().splitlines()
lang_df = pd.DataFrame({"text": texts, "lang": labels})

# шесть языков
LANGS = {"kaz": "kk", "rus": "ru", "eng": "en", "spa": "es", "por": "pt", "deu": "de"}
lang_df = lang_df[lang_df["lang"].isin(LANGS)]
lang_df["lang"] = lang_df["lang"].map(LANGS)

# удаляем дубликаты
lang_df = lang_df.drop_duplicates(subset="text")
print(lang_df["lang"].value_counts())

# 80 / 20
train_lang, test_lang = train_test_split(lang_df, test_size=0.20, stratify=lang_df["lang"], random_state=SEED)
train_lang, test_lang = train_lang.copy(), test_lang.copy()
print("\ntrain:", len(train_lang), "| test:", len(test_lang))

### Шаг 2. Эксперимент с предварительной обработкой

Сравнить две стратегии предварительной обработки.

Подход A — Интенсивная предварительная обработка:
- Привести тексты к нижнему регистру.
- Удалить цифры.
- Удалить знаки препинания и специальные символы.

Подход B — Минимальная предварительная обработка:
- Сохранить символы, характерные для отдельных языков.
- Удалить только лишние пробелы и артефакты форматирования.

Объяснить влияние предварительной обработки на распознавание казахского и других языков, содержащих специфические символы.

Важно: оба подхода должны сохранять допустимые буквы Unicode. Не допускается удаление специфических букв казахского языка, таких как ə, ғ, қ, ң, ө, ұ, ү, һ и і.

In [ ]:
# A: нижний регистр, без цифр и знаков; isalpha() оставляет буквы всех алфавитов
def preprocess_A(text):
    text = text.lower()
    text = "".join(c if c.isalpha() or c.isspace() else " " for c in text)
    return " ".join(text.split())

# B: только лишние пробелы
def preprocess_B(text):
    return " ".join(text.split())

for part in [train_lang, test_lang]:
    part["A"] = part["text"].apply(preprocess_A)
    part["B"] = part["text"].apply(preprocess_B)

# проверка: казахские буквы на месте
example = train_lang[train_lang["lang"] == "kk"].iloc[0]
print("Исходный:", example["text"][:120])
print("A:", example["A"][:120])
print("B:", example["B"][:120])

**Влияние обработки на казахский.** В обоих подходах казахские буквы (ә, ғ, қ, ң, ө, ұ, ү, һ, і) сохраняются — это видно в примере. `isalpha()` считает буквой любой символ алфавита, не только латиницу. Если бы символы удалялись шаблоном `[^a-z]`, казахский и русский текст исчез бы полностью.

Эти буквы — главный признак, по которому казахский отличается от русского: кириллица у них общая, но ә, қ, ң, ө, ұ, ү, і есть только в казахском. Подход A ещё и приводит текст к нижнему регистру — для определения языка регистр почти не важен.

### Шаг 3. Представление текстовых признаков

Реализовать два способа представления текста:

1. TF-IDF на уровне слов (Word-level TF-IDF).
2. TF-IDF на уровне символов с использованием символьных n-грамм (1–3).

Сравнить эффективность обоих подходов для задачи идентификации языка.

### Шаг 4. Разработка моделей

Обучить следующие классификаторы:

- Multinomial Naive Bayes.
- Logistic Regression.

Сравнить обученные модели с предварительно обученной системой определения языка, например fastText lid.176.

Предварительно обученную модель необходимо оценивать на той же отложенной тестовой выборке.

In [ ]:
# все комбинации: обработка (A, B) x признаки (слова, символы) x модель (NB, LR)
rows = []
trained = {}
for prep in ["A", "B"]:
    for feat_name in ["слова", "символы 1-3"]:
        if feat_name == "слова":
            vectorizer = TfidfVectorizer(analyzer="word")
        else:
            vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(1, 3))
        X_train = vectorizer.fit_transform(train_lang[prep])
        X_test = vectorizer.transform(test_lang[prep])
        for model_name, model in [("Naive Bayes", MultinomialNB()), ("Logistic Regression", LogisticRegression(max_iter=1000))]:
            model.fit(X_train, train_lang["lang"])
            pred = model.predict(X_test)
            rows.append({
                "обработка": prep,
                "признаки": feat_name,
                "модель": model_name,
                "accuracy": accuracy_score(test_lang["lang"], pred),
                "macro-F1": f1_score(test_lang["lang"], pred, average="macro"),
            })
            trained[(prep, feat_name, model_name)] = (vectorizer, model, pred)

results = pd.DataFrame(rows).round(4)
results

In [ ]:
# предобученная модель fastText lid.176 на той же тестовой выборке
import fasttext

if not os.path.exists("lid.176.bin"):
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin", "lid.176.bin")
ft = fasttext.load_model("lid.176.bin")

def fasttext_predict(text):
    label = ft.predict(text.replace("\n", " "))[0][0]
    return label.replace("__label__", "")

ft_pred = test_lang["text"].apply(fasttext_predict)
print("fastText | accuracy =", round(accuracy_score(test_lang["lang"], ft_pred), 4),
      "| macro-F1 =", round(f1_score(test_lang["lang"], ft_pred, average="macro", labels=list(LANGS.values())), 4))

### Шаг 5. Оценка качества моделей

Рассчитать:

- Accuracy.
- Macro F1-score.
- Confusion Matrix.

Сравнить результаты для различных стратегий предварительной обработки, способов представления признаков и моделей классификации.

Определить наиболее эффективную конфигурацию.

In [ ]:
# лучшая конфигурация по macro-F1
best = results.sort_values("macro-F1", ascending=False).iloc[0]
best_key = (best["обработка"], best["признаки"], best["модель"])
best_vectorizer, best_model, best_pred = trained[best_key]
print("Лучшая конфигурация:", best_key)

ConfusionMatrixDisplay.from_predictions(test_lang["lang"], best_pred, colorbar=False)
plt.title(" + ".join(best_key))
plt.show()

**Сравнение:**
- Все 8 вариантов дают accuracy 0,981–0,988 — на длинных абзацах разница маленькая.
- **Лучшая конфигурация:** подход B + символьные n-граммы 1–3 + Logistic Regression — accuracy 0,988, macro-F1 0,988.
- Logistic Regression везде лучше Naive Bayes.
- Признаки на словах и символах на полных текстах почти равны (0,987 и 0,988).
- Обработка A или B почти не влияет.
- fastText lid.176 без обучения на наших данных даёт accuracy 0,983 и macro-F1 0,985 — почти столько же, хотя выбирает из 176 языков.

### Шаг 6. Тестирование устойчивости модели (Robustness Testing)

#### Эксперимент A — Короткие тексты

Сравнить качество определения языка для текстов различной длины:
- 1–3 слова.
- 4–10 слов.
- Более 10 слов.

In [ ]:
# короткие тексты: первые 3 и первые 10 слов тестовых абзацев
lengths = {
    "1-3 слова": test_lang["B"].apply(lambda t: " ".join(t.split()[:3])),
    "4-10 слов": test_lang["B"].apply(lambda t: " ".join(t.split()[:10])),
    "больше 10 слов": test_lang["B"],
}

rows = []
for name, texts in lengths.items():
    row = {"длина": name}
    for feat_name in ["слова", "символы 1-3"]:
        vectorizer, model, _ = trained[("B", feat_name, "Logistic Regression")]
        row["LR, " + feat_name] = accuracy_score(test_lang["lang"], model.predict(vectorizer.transform(texts)))
    rows.append(row)
pd.DataFrame(rows).round(4)

#### Эксперимент B — Тексты с ошибками

Создать модифицированную версию тестовой выборки, добавив контролируемые орфографические ошибки:
- Удаление символов.
- Вставка дополнительных символов.
- Замена символов.

Сравнить качество классификации до и после внесения ошибок.

In [ ]:
rng = random.Random(SEED)

# с вероятностью 10% каждая буква удаляется, получает лишнюю букву или заменяется
def add_typos(text):
    letters = [c for c in text if c.isalpha()] or ["a"]
    result = ""
    for c in text:
        r = rng.random()
        # удаление
        if r < 0.1:
            continue
        # вставка лишней буквы
        elif r < 0.2:
            result += c + rng.choice(letters)
        # замена
        elif r < 0.3 and c.isalpha():
            result += rng.choice(letters)
        else:
            result += c
    return result

noisy = test_lang["text"].apply(add_typos)
print("Пример:", noisy.iloc[0][:120])

prep = best_key[0]
noisy = noisy.apply(preprocess_A if prep == "A" else preprocess_B)
noisy_pred = best_model.predict(best_vectorizer.transform(noisy))
print("\nДо ошибок: accuracy =", round(accuracy_score(test_lang["lang"], best_pred), 4))
print("После ошибок: accuracy =", round(accuracy_score(test_lang["lang"], noisy_pred), 4))

#### Эксперимент C — Смешение языков (Code-Switching)

- Составить не менее 30 предложений со смешением казахского и русского языков.
- Проанализировать предсказания обученной модели для смешанных текстов.
- Объяснить, почему классификатор с одной языковой меткой может некорректно обрабатывать смешение языков.

In [ ]:
mixed = [
    "Сәлеметсіз бе, как дела?",
    "Мен сағат беске кофе ішемін, потом иду на работу.",
    "Бұл есеп очень сложный болды.",
    "Таңертең жүгірдім, а вечером отдыхал.",
    "Қаладағы движение қатты интенсивный.",
    "Маған кофе қажет, срочно.",
    "Ол концертке барды, но мне не понравилось.",
    "Бұл кітап интересная, прочитал за ночь.",
    "Жаңалықтарды қарадым, ничего нового.",
    "Концерт прошёл сәтті.",
    "Апам пирог жасады, вкусно получилось.",
    "Университетте лекция болды, по математике.",
    "Қалай ойлайсың, стоит ли идти?",
    "Мен бүгін тойдым, не могу больше есть.",
    "Ол машинамен келді, прямо к подъезду.",
    "Жаңа фильм көрдім, супер.",
    "Қазір сөйлесе алмаймын, перезвоню.",
    "Жұмыстан кейін кездесеміз, в кафе.",
    "Ол ағылшынша говорит жақсы.",
    "Бұл есеп тяжёлый, нужно время.",
    "Жаңа үй сатып алдым, ипотека на 15 лет.",
    "Мен бүгін тойға барамын, в семь вечера.",
    "Апам пирог пісірді, с яблоками.",
    "Жаңа жоба басталды, срочный.",
    "Университет жабық, карантин жарияланды.",
    "Концерт жақсы болды, зал был полный.",
    "Жаңа планшет сатып алдым, для работы.",
    "Жаңбыр жауды, промок до нитки.",
    "Мен бүгін ертерек тұрдым, в шесть утра.",
    "Жаңа курс басталды, по программированию.",
]

# предсказание и уверенность (максимальная вероятность)
X_mixed = best_vectorizer.transform([preprocess_A(m) if prep == "A" else preprocess_B(m) for m in mixed])
mixed_df = pd.DataFrame({
    "предложение": mixed,
    "язык": best_model.predict(X_mixed),
    "уверенность": best_model.predict_proba(X_mixed).max(axis=1).round(2),
})
print(mixed_df["язык"].value_counts())
mixed_df

**A. Длина текста** (accuracy, Logistic Regression):

| длина | слова | символы 1–3 |
|---|---|---|
| 1–3 слова | 0,667 | 0,803 |
| 4–10 слов | 0,916 | 0,964 |
| больше 10 слов | 0,987 | 0,988 |

Чем короче текст, тем хуже. На коротких текстах символьные признаки сильно лучше: +14 п.п. на 1–3 словах.

**B. Опечатки.** Точность упала с 0,988 до 0,981 — символьная модель устойчива к опечаткам: испорченная буква меняет лишь несколько n-грамм, остальные остаются.

**C. Смешение языков.**
- Из 30 смешанных предложений модель назвала 18 казахскими и 12 русскими.
- Уверенность низкая: 0,44–0,80, в большинстве предложений около 0,5–0,7.

Классификатор с одной меткой обязан выбрать один язык, хотя в предложении их два. Он выбирает тот, чьих признаков больше: в «Мен бүгін тойдым, не могу больше есть» перевесил казахский, в «Концерт прошёл сәтті» — русский. Модель обучалась только на одноязычных текстах и смешанных не видела.

### Шаг 7. Анализ ошибок

- Найти не менее пяти неправильных предсказаний.
- Проанализировать влияние длины текста, общей лексики, орфографических ошибок и близкородственных языков.
- Объяснить, почему символьные признаки могут превосходить признаки на уровне слов при обработке коротких многоязычных текстов.
- Предложить способы повышения надёжности систем автоматического определения языка.

Исследовательский вопрос: как представление признаков, предварительная обработка и качество текста влияют на эффективность многоязычной идентификации, особенно для казахского и русского языков?

In [ ]:
# ошибки лучшей модели на полных текстах
errors = test_lang[best_pred != test_lang["lang"]][["text", "lang"]]
errors["предсказано"] = best_pred[best_pred != test_lang["lang"]]
errors["text"] = errors["text"].str[:100]
print("Ошибок на полных текстах:", len(errors))
errors.head(10)

In [ ]:
# ошибки на текстах из 1-3 слов (LR на символах)
vectorizer, model, _ = trained[("B", "символы 1-3", "Logistic Regression")]
short_pred = model.predict(vectorizer.transform(lengths["1-3 слова"]))
short_errors = pd.DataFrame({"текст": lengths["1-3 слова"], "верно": test_lang["lang"], "предсказано": short_pred})
short_errors = short_errors[short_errors["верно"] != short_errors["предсказано"]]
print("Ошибок на текстах 1-3 слова:", len(short_errors), "из", len(test_lang))
short_errors.head(10)

**Ошибки на полных текстах** — 14 из 1185. Из 10 показанных:
- 8 — это английские библиографические ссылки внутри португальских и русских статей (`Farrington, D. P. (1994b)...`, `Thomson, Harry C. ...`). Модель ответила `en` и по сути права — это ошибка разметки датасета.
- `de → es`: немецкий текст с французским названием («Six sonates en trio»).
- `de → pt`: немецкий текст из португальских названий городов.

**Ошибки на текстах из 1–3 слов** — 233 из 1185:
- **Общая лексика:** даты и числа («On September 12», «Am 5. Oktober», «From 5 September») — `en` и `de` путаются между собой и с `pt`.
- **Близкие языки:** «En febrero de» (es → de).
- **Казахский ↔ русский:** «Профессор Бадаев С.А.», «Курт Дональд Кобейн» в казахских текстах — имена пишутся одинаково, казахских букв нет.

**Опечатки** снизили точность всего на 0,7 п.п. (0,988 → 0,981).

**Почему символьные признаки лучше на коротких текстах.** В 1–3 словах часто нет ни одного слова, которое модель видела при обучении, и признаки на уровне слов пусты. Символьные n-граммы есть в любом слове: `қ`, `ің`, `ção`, `sch`, `ий` сразу указывают на язык.

**Как повысить надёжность:**
- использовать символьные n-граммы или комбинацию слов и символов;
- добавить в обучение короткие тексты;
- при низкой уверенности отвечать «не определено»;
- для смешанных текстов определять язык по словам, а не для всего предложения;
- чистить данные от вставок на других языках.

**Ответ на исследовательский вопрос.**
- На длинных текстах признаки и обработка почти не влияют — около 98–99%.
- Решает качество текста: короткие тексты снижают точность до 67–80%.
- Символьные n-граммы лучше всего справляются с короткими текстами и опечатками.
- Для казахского и русского главное — сохранить специфические буквы казахского. Без них (в именах, коротких фразах) модели путают эти два языка.